In [ ]:
# CELL 1 — GPU + Drive mount
import torch, shutil, os
from google.colab import drive

assert torch.cuda.is_available(), "Switch to T4 GPU first"
print(f"GPU: {torch.cuda.get_device_name(0)}")
total, used, free = shutil.disk_usage('/')
assert free > 40*1024**3, f"Only {free//1024**3}GB free — need 40GB+"
print(f"Disk: {free//1024**3}GB free")

drive.mount('/content/drive')
DRIVE_SAVE = '/content/drive/MyDrive/PlantoAI_Models'
os.makedirs(DRIVE_SAVE, exist_ok=True)
print(f"Drive mounted. Models → {DRIVE_SAVE}")

GPU: Tesla T4
Disk: 69GB free


In [ ]:
# CELL 2 — Install correct dependencies
!pip install open-clip-torch timm kaggle datasets \
    huggingface_hub albumentations imagededup tqdm -q
print("Dependencies ready")

In [ ]:
# CELL 3 — Kaggle credentials
import json, os
# Fill these in:
KAGGLE_USER = "hackrore"
KAGGLE_KEY  = "your_key_here"

os.makedirs("/root/.kaggle", exist_ok=True)
with open("/root/.kaggle/kaggle.json","w") as f:
    json.dump({"username": KAGGLE_USER, "key": KAGGLE_KEY}, f)
os.chmod("/root/.kaggle/kaggle.json", 0o600)
print("Kaggle credentials set")

In [ ]:
# CELL 4 — Download datasets directly to VM
import subprocess
from pathlib import Path
from datasets import load_dataset
from uuid import uuid4
from tqdm.auto import tqdm
import requests

RAW = Path("/content/raw")
RAW.mkdir(exist_ok=True)

# Kaggle
for ds_id, name in [
    ("satyamtomar08/indian-medicinal-plant-dataset","cimpd"),
    ("abdallahalidev/plantvillage-dataset","plantvillage"),
    ("mdfahimbinalam/leaf-dataset","leaf_dataset"),
]:
    out = RAW/name
    out.mkdir(exist_ok=True)
    r = subprocess.run(
        ["kaggle","datasets","download","-d",ds_id,"-p",str(out),"--unzip"],
        capture_output=True, text=True)
    print(f"{name}: {r.stdout.strip() or r.stderr.strip()}")

# HuggingFace
try:
    ds = load_dataset(
        "PROFESSOR-DJ/Indian_Medicinal_Plants_and_Leaf_Dataset_Large",
        split="train")
    for item in tqdm(ds, desc="HuggingFace"):
        label = str(item.get("label", item.get("plant_name","unknown")))
        out = RAW/"huggingface"/label
        out.mkdir(parents=True, exist_ok=True)
        item["image"].save(out/f"{uuid4()}.jpg")
except Exception as e:
    print(f"HF warning: {e}")

# iNaturalist — 20 priority Indian species
SPECIES = [
    "Withania somnifera","Tinospora cordifolia","Phyllanthus emblica",
    "Terminalia chebula","Bacopa monnieri","Centella asiatica",
    "Curcuma longa","Glycyrrhiza glabra","Asparagus racemosus",
    "Tribulus terrestris","Mucuna pruriens","Boswellia serrata",
    "Piper longum","Nardostachys jatamansi","Acorus calamus",
]
for sp in tqdm(SPECIES, desc="iNaturalist"):
    out = RAW/"inaturalist"/sp.replace(" ","_")
    out.mkdir(parents=True, exist_ok=True)
    try:
        r = requests.get("https://api.inaturalist.org/v1/observations",
            params={"taxon_name":sp,"quality_grade":"research",
                    "photos":"true","per_page":100}, timeout=15).json()
        for obs in r.get("results",[]):
            try:
                url = obs["photos"][0]["url"].replace("square","large")
                (out/f"{uuid4()}.jpg").write_bytes(
                    requests.get(url,timeout=10).content)
            except: continue
    except: continue

# Count
total = sum(1 for _ in RAW.rglob("*.jpg"))
print(f"\nTotal images downloaded: {total:,}")

In [ ]:
# CELL 5 — Merge: resolve synonyms + quality gate + split
import shutil
from pathlib import Path
from PIL import Image

RAW = Path("/content/raw")
MERGED = Path("/content/merged")

SYNONYMS = {
    "tulsi":"ocimum_tenuiflorum","holy_basil":"ocimum_tenuiflorum",
    "neem":"azadirachta_indica","indian_lilac":"azadirachta_indica",
    "brahmi":"bacopa_monnieri","amla":"phyllanthus_emblica",
    "giloy":"tinospora_cordifolia","ashwagandha":"withania_somnifera",
}

species_images = {}
for img_path in tqdm(list(RAW.rglob("*.jpg")), desc="Processing"):
    raw_name = img_path.parent.name.lower().replace(" ","_").replace("-","_")
    canonical = SYNONYMS.get(raw_name, raw_name)
    try:
        img = Image.open(img_path).convert("RGB")
        w, h = img.size
        if w < 100 or h < 100: continue  # too small
        species_images.setdefault(canonical, []).append(img_path)
    except: continue

# Keep species with >= 80 images
TRAIN = MERGED/"train"; VAL = MERGED/"val"; TEST = MERGED/"test"
kept = 0
for sp, paths in species_images.items():
    if len(paths) < 80:
        print(f"Skip {sp}: only {len(paths)} images")
        continue
    import random; random.shuffle(paths)
    n = len(paths)
    splits = [("train", paths[:int(n*0.75)]),
              ("val",   paths[int(n*0.75):int(n*0.9)]),
              ("test",  paths[int(n*0.9):])]
    for split_name, split_paths in splits:
        out = MERGED/split_name/sp
        out.mkdir(parents=True, exist_ok=True)
        for p in split_paths:
            shutil.move(p, out/p.name)
    kept += 1

# FREE DISK SPACE - Crucial for Colab Free Tier
shutil.rmtree(RAW, ignore_errors=True)
shutil.rmtree("/root/.cache/huggingface", ignore_errors=True)

classes = list((MERGED/"train").iterdir())
print(f"\n✅ {kept} species ready | {len(classes)} classes")
print(f"Train: {sum(1 for _ in (MERGED/'train').rglob('*.jpg')):,} images")
assert kept > 0, "FATAL: No species passed the 80-image threshold"

In [ ]:
# CELL 6 — BioCLIP 2 model — CORRECT implementation
import open_clip
import torch
import torch.nn as nn
from pathlib import Path

MERGED = Path("/content/merged")
classes = sorted([d.name for d in (MERGED/"train").iterdir()])
NUM_CLASSES = len(classes)
print(f"Training on {NUM_CLASSES} species")

# Save class names immediately — needed for deployment
import json
class_names_path = "/content/class_names.json"
with open(class_names_path, "w") as f:
    json.dump(classes, f, indent=2)
print(f"Saved {NUM_CLASSES} class names")

DEVICE = torch.device("cuda")

# Load TRUE BioCLIP 2 — NOT a timm ViT
backbone, _, preprocess = open_clip.create_model_and_transforms(
    "hf-hub:imageomics/bioclip-2"
)
backbone = backbone.to(DEVICE)

# FREEZE all backbone parameters
for p in backbone.parameters():
    p.requires_grad = False

# UNFREEZE last 4 transformer blocks only
visual = backbone.visual
if hasattr(visual, "transformer"):
    blocks = visual.transformer.resblocks
    for block in blocks[-4:]:
        for p in block.parameters():
            p.requires_grad = True

# Classification head on top of 512-dim embedding
embed_dim = backbone.visual.output_dim
class BotanicalClassifier(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone = backbone
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, 256),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(256, NUM_CLASSES)
        )
    def forward(self, x):
        with torch.cuda.amp.autocast():
            feat = self.backbone.encode_image(x)
        return self.head(feat.float())

model = BotanicalClassifier().to(DEVICE)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,} (head + last 4 blocks only)")

In [ ]:
# CELL 7 — Dataloaders with CORRECT BioCLIP 2 normalization
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# BioCLIP 2 normalization — NOT ImageNet values
BIOCLIP_MEAN = [0.48145466, 0.4578275,  0.40821073]
BIOCLIP_STD  = [0.26862954, 0.26130258, 0.27577711]
IMG_SIZE = 384

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(IMG_SIZE, scale=(0.7,1.0)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.3,0.3,0.3,0.1),
    transforms.ToTensor(),
    transforms.Normalize(BIOCLIP_MEAN, BIOCLIP_STD),
])
val_tf = transforms.Compose([
    transforms.Resize(IMG_SIZE+32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(BIOCLIP_MEAN, BIOCLIP_STD),
])

train_ds = datasets.ImageFolder(str(MERGED/"train"), transform=train_tf)
val_ds   = datasets.ImageFolder(str(MERGED/"val"),   transform=val_tf)

# WeightedRandomSampler for class imbalance
counts  = torch.tensor([train_ds.targets.count(i)
                        for i in range(NUM_CLASSES)], dtype=torch.float)
weights = 1.0 / counts[train_ds.targets]
sampler = torch.utils.data.WeightedRandomSampler(weights, len(weights))

BATCH = 16  # T4 safe with ViT-L/14 at 384px
train_dl = DataLoader(train_ds, batch_size=BATCH, sampler=sampler,
                      num_workers=2, pin_memory=True)
val_dl   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,
                      num_workers=2, pin_memory=True)
print(f"Train: {len(train_ds):,} | Val: {len(val_ds):,} | Batches: {len(train_dl):,}")

In [ ]:
# CELL 8 — Training loop — saves BEST model, not last
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR

EPOCHS = 30
CHECKPOINT_DIR = Path("/content/checkpoints")
CHECKPOINT_DIR.mkdir(exist_ok=True)

# Differential learning rates
optimizer = optim.AdamW([
    {"params": model.head.parameters(),     "lr": 1e-4},
    {"params": [p for p in model.backbone.parameters()
                if p.requires_grad],        "lr": 1e-5},
], weight_decay=1e-4)
scheduler  = CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-7)
criterion  = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler     = torch.cuda.amp.GradScaler()

best_val_acc = 0.0
log = []

for epoch in range(1, EPOCHS+1):
    # Train
    model.train()
    train_loss = 0.0
    for imgs, labels in train_dl:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast():
            loss = criterion(model(imgs), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update()
        train_loss += loss.item()

    # Validate
    model.eval()
    correct = total = top3_correct = 0
    with torch.no_grad():
        for imgs, labels in val_dl:
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            logits = model(imgs)
            top3   = logits.topk(3,dim=1).indices
            correct      += (logits.argmax(1)==labels).sum().item()
            top3_correct += sum(labels[i] in top3[i] for i in range(len(labels)))
            total        += len(labels)

    val_acc  = correct/total
    top3_acc = top3_correct/total
    avg_loss = train_loss/len(train_dl)
    scheduler.step()
    log.append({"epoch":epoch,"loss":avg_loss,
                "val_acc":val_acc,"top3_acc":top3_acc})
    print(f"Epoch {epoch:02d}/{EPOCHS} | "
          f"Loss:{avg_loss:.4f} | Val:{val_acc:.4f} | Top3:{top3_acc:.4f}")

    # Save BEST checkpoint only
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(),
                   CHECKPOINT_DIR/"best_model.pt")
        print(f"  ✓ New best: {val_acc:.4f}")

    # Save every 5 epochs as insurance
    if epoch % 5 == 0:
        torch.save(model.state_dict(),
                   CHECKPOINT_DIR/f"epoch_{epoch:02d}.pt")

with open(CHECKPOINT_DIR/"training_log.json","w") as f:
    json.dump(log, f, indent=2)
print(f"\nDone. Best val accuracy: {best_val_acc:.4f}")

In [ ]:
# CELL 9 — Save to Drive + verification
import shutil, datetime
from pathlib import Path

CHECKPOINT_DIR = Path("/content/checkpoints")
best = CHECKPOINT_DIR/"best_model.pt"

assert best.exists(), "FATAL: best_model.pt missing — training failed"
size_mb = best.stat().st_size / 1024**2
print(f"best_model.pt size: {size_mb:.0f}MB")

ts = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
save_dir = Path(f"/content/drive/MyDrive/PlantoAI_Models/run_{ts}")
save_dir.mkdir(parents=True, exist_ok=True)

shutil.copy(best, save_dir/"best_model.pt")
shutil.copy("/content/class_names.json", save_dir/"class_names.json")
shutil.copy(CHECKPOINT_DIR/"training_log.json", save_dir/"training_log.json")

print(f"✅ Saved to Drive: {save_dir}")
print(f"✅ {NUM_CLASSES} species | Best accuracy: {best_val_acc:.4f}")
print("\nNEXT: Download best_model.pt + class_names.json from Drive")
print("Place at: scaling/artifacts/checkpoints/best_model.pt")
print("Tell Antigravity: Run Phase 4 ONNX export")